# ryGPT — Kaggle T4 training

Fine-tune Qwen2.5-1.5B with QLoRA on your anonymized WhatsApp data.

## Before you run

In the right-sidebar of this notebook, confirm:

1. **Accelerator: GPU T4 ×1**
2. **Internet: On** (needed to download Qwen weights)
3. **Add Data** → your private `rygpt-data` dataset (contains `train.jsonl`, `val.jsonl`, `name_mapping.json`)

If any of these are missing, edit the notebook settings first — otherwise the cells below will fail with a clear error message.

## 1. Environment check

In [ ]:
!nvidia-smi | head -20
import torch
print()
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('Device:', torch.cuda.get_device_name(0))
    print('Compute capability:', cap)
    print(f'Training precision: {"bf16" if cap[0] >= 8 else "fp16"}')
else:
    raise SystemExit('No GPU detected. Settings → Accelerator → GPU T4 ×1')

## 2. Verify internet is on

Without internet, we can't download the base model.

In [ ]:
import urllib.request
try:
    urllib.request.urlopen('https://github.com', timeout=5)
    print('Internet: OK')
except Exception as e:
    raise SystemExit(
        'Internet appears OFF. Settings → Internet → On. Then re-run this cell.\n'
        f'Error: {e}'
    )

## 3. Clone the repo

Idempotent — safe to re-run if the previous attempt failed.

In [ ]:
import os, shutil
REPO_DIR = '/kaggle/working/ryGPT'
if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} already exists — removing and re-cloning for a clean state')
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/rihaans/ryGPT.git {REPO_DIR}
%cd {REPO_DIR}
!ls

## 4. Install dependencies

Kaggle images already have `torch`/`transformers`. We add `bitsandbytes` for 4-bit quantization and pin the ryGPT `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes
# Verify
import bitsandbytes as bnb
from peft import LoraConfig
print('bitsandbytes:', bnb.__version__)
print('peft imported OK')

## 5. Wire the Kaggle dataset into the expected paths

Auto-detects any `/kaggle/input/*/train.jsonl` so it works regardless of what you named your dataset slug.

In [ ]:
import glob, os, shutil

candidates = glob.glob('/kaggle/input/*/train.jsonl')
if not candidates:
    raise SystemExit(
        'Could not find train.jsonl in /kaggle/input/. Add your `rygpt-data` dataset\n'
        '(train.jsonl + val.jsonl + name_mapping.json) via the Add Data button in the sidebar.'
    )
dataset_dir = os.path.dirname(candidates[0])
print(f'Found dataset at: {dataset_dir}')

os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/anonymized', exist_ok=True)

for src_name, dst_path in [
    ('train.jsonl', 'data/processed/train.jsonl'),
    ('val.jsonl', 'data/processed/val.jsonl'),
    ('name_mapping.json', 'data/anonymized/name_mapping.json'),
]:
    src = os.path.join(dataset_dir, src_name)
    if not os.path.exists(src):
        print(f'  MISSING in dataset: {src_name}  (optional: name_mapping.json)')
        continue
    shutil.copy(src, dst_path)
    print(f'  {src_name}  ({os.path.getsize(src)/1e6:.1f} MB)  ->  {dst_path}')

!ls -lh data/processed data/anonymized

## 6. Sanity-check the data

Confirms the training script can read the JSONL correctly before we spend hours training.

In [ ]:
import sys, json
sys.path.insert(0, '/kaggle/working/ryGPT')
from src.dataset import read_jsonl, example_to_chat_messages

train = read_jsonl('data/processed/train.jsonl')
val = read_jsonl('data/processed/val.jsonl')
print(f'train: {len(train):,}  |  val: {len(val):,}')
print()
print('First train example (chat template):')
for m in example_to_chat_messages(train[0]):
    print(f'  [{m["role"]:>9}] {m["content"][:100]}')

## 7. Train (Phase 6)

T4-tuned hyperparameters:
- `--batch-size 8 --grad-accum 2` → effective batch 16 (fits in 15 GB VRAM)
- `--max-seq-length 256` (p99 of real data is 170 tokens — clips only 0.16%)
- `--epochs 2` (safe fit within 12-hour Kaggle session; bump to 3 if you want)

Expected wall time: **~5-6 hours** on a T4. Log lines print every 100 steps; eval every 1000.

**Idle timeout:** Kaggle kills idle sessions after ~20 min. Either keep the tab open, or click **Save Version → Save & Run All** (top-right) to run headless.

In [ ]:
!python scripts/06_train_model.py \
    --base-model Qwen/Qwen2.5-1.5B \
    --batch-size 8 \
    --grad-accum 2 \
    --max-seq-length 256 \
    --epochs 2 \
    --eval-steps 1000 \
    --save-steps 1000 \
    --logging-steps 100 \
    --wandb-disabled \
    --out-dir /kaggle/working/ryGPT/models/lora_adapter

## 8. Evaluate (Phase 7)

Produces `eval/perplexity.md`, `eval/samples.md`, `eval/memorization.md`, and (if you added negative-class corpora) `eval/style_classifier.md`.

Expected wall time: **~45 min** on T4.

In [ ]:
!python scripts/07_evaluate.py

## 9. Preview eval results inline

In [ ]:
from pathlib import Path
for name in ('perplexity', 'memorization', 'samples', 'style_classifier'):
    p = Path(f'/kaggle/working/ryGPT/eval/{name}.md')
    if p.exists():
        print(f'{"=" * 20} eval/{name}.md {"=" * 20}')
        # cap at 4000 chars to keep the notebook readable
        text = p.read_text(encoding='utf-8')
        print(text[:4000])
        if len(text) > 4000:
            print(f'\n... ({len(text) - 4000} more chars in the file)')
        print()

## 10. Package adapter for download

Bundles the LoRA adapter + tokenizer + eval markdown into a single archive under `/kaggle/working/` — which Kaggle exposes as **Output** files you can download from the right sidebar.

In [ ]:
!cd /kaggle/working/ryGPT && tar czf /kaggle/working/rygpt_lora_adapter.tar.gz models/lora_adapter eval
!ls -lh /kaggle/working/rygpt_lora_adapter.tar.gz

## 11. Download to your laptop

1. In the right sidebar of this notebook, expand **Output**.
2. Find `rygpt_lora_adapter.tar.gz` and click the download icon.
3. On your laptop:

```powershell
cd C:\Users\rihaa\Development\Projects\ryGPT
tar -xzf rygpt_lora_adapter.tar.gz
# Extracts models/lora_adapter/ and eval/*.md
python scripts/chat.py
```

Inference runs fine on your 4070 laptop — no GPU rental needed past this point.

## Stop the session

When done, hit **Stop** in the top-right to release the T4 back to your 30-hour weekly quota.